In [ ]:
import arviz as az
import pymc as pm
import numpy as np
from matplotlib import pyplot as plt

# Convergence and sample size estimation in PyMC3


In [ ]:
D = [1]*2 + [0] * 1

with pm.Model() as model1:
    theta1 = pm.Beta (name = "theta1", alpha = 1, beta = 1)
    y = pm.Bernoulli (name = "obs", p = theta1, observed = D)

In [ ]:
with model1:
    sample = pm.sample(
            500, # increase for more trustworthy results (we show small numbers to visualize the impact of the initial seed)
            step=pm.step_methods.metropolis.Metropolis(S=np.array([0.2])),
            chains = 3,  # we only plot 1 chain (normally 4 is recommended)
            start = [{ "theta1": 0.01 }, { "theta1": 0.01 }, { "theta1": 0.99 }], # 3 starting points
            discard_tuned_samples = True, # Show the burn-in prefix
            tune = 500  
    );

* Flip 'discard_tuned_samples' to True, to show a more representative (trustworthy) density plot (to the left)
* You should never do any real inference without discarding the tuning samples (True).  We only do this for learning purpose here.
* The prefix before convergence is called “burn-in period”, controlled by the tune parameter in pymc3 (500 is default, you may want to increase this value, if you can see that 500 is not enough time for the chains to overlap)
* Despite of what the documentation says, 'tune' apparently also works for Metropolis, not just for NUTS. Set it to zero to see how bad sampling is without burn-in


In [ ]:
with model1:
  ax = az.plot_trace(sample, kind="rank_bars")
  ax[0,0].axis([0.0, 1.0, 0.0, 7.0])
  # clip the trace plot (to the right, to show the entire development better)
  # ax[0,1].axis([0,100,0.0, 1.0,]) 
  plt.show()

In [ ]:
with model1:
  az.plot_autocorr(sample)
  plt.show()

Autocorrelation approaches zero for all three chains (as we would like it to)

In [ ]:
with model1:
    df = az.summary (sample)
df

* I don't understand __all__ the above stats, but mcse, ess_bulk, ess_tail, and r_hat have been discussed in the book
* __ess_tail__ is relevant if you estimate the __HDI endpoints__
* __ess_bulk__ is relevant for the __mean__ (but mean does not require a lot of samples, 200 are probably good, especially for symmetric distributions).